In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import ( col, when, trim, lit, regexp_replace, split, explode, lower, array_contains, transform, isnan, count, round)
from pyspark.sql.types import IntegerType
from delta import *

warehouse_location = 'hdfs://hdfs-nn:9000/projeto/silver'

builder = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("ETL-Amazon_books-Silver") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("SparkSession iniciada com sucesso e Delta Lake configurado.")

SparkSession iniciada com sucesso e Delta Lake configurado.


In [2]:
hdfs_path = "hdfs://hdfs-nn:9000/projeto/bronze/amazon_books.csv"

In [3]:
df_bronze = spark.read.csv(
    hdfs_path,
    header=True,
    inferSchema=True,
    quote='"',
    escape='"',
    multiLine=True
)

In [4]:
df_bronze = df_bronze.drop("timestamp")

In [5]:
print(f"Total de registos lidos da camada Bronze: {df_bronze.count()}")
df_bronze.printSchema()
df_bronze.toPandas()

Total de registos lidos da camada Bronze: 2269
root
 |-- asin: string (nullable = true)
 |-- ISBN10: string (nullable = true)
 |-- title: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- availability: string (nullable = true)
 |-- discount: double (nullable = true)
 |-- initial_price: double (nullable = true)
 |-- final_price: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- reviews_count: integer (nullable = true)
 |-- categories: string (nullable = true)
 |-- best_sellers_rank: string (nullable = true)
 |-- url: string (nullable = true)



,asin,ISBN10,title,brand,availability,discount,initial_price,final_price,currency,rating,reviews_count,categories,best_sellers_rank,url
0,0007350813,0007350813,Wuthering Heights (Collins Classics),Emily Brontë,In Stock.,NaN,NaN,3.99,USD,4.6 out of 5 stars,13451,"[""Books"",""Literature & Fiction"",""Genre Fiction""]","[{""category"":""Books / Literature & Fiction / H...",https://www.amazon.com/dp/0007350813
1,0007513763,9780007513765,THE DAYS THE CRAYONS QUIT,Drew Daywalt,In Stock.,NaN,NaN,12.08,USD,4.8 out of 5 stars,16628,"[""Books"",""Children's Books"",""Literature & Fict...","[{""category"":""Books / Children's Books / Liter...",https://www.amazon.com/dp/0007513763
2,0008183988,0008183988,War Lord: Book 13 (The Last Kingdom Series),Bernard Cornwell,None,NaN,NaN,NaN,USD,4.8 out of 5 stars,11275,"[""Books"",""Literature & Fiction"",""Genre Fiction""]","[{""category"":""Books / Literature & Fiction / H...",https://www.amazon.com/dp/0008183988
3,0008305838,0008305838,Code Name Bananas: The hilarious and epic new ...,David Walliams,In Stock.,NaN,NaN,20.43,USD,4.8 out of 5 stars,15520,"[""Books"",""Children's Books"",""Literature & Fict...","[{""category"":""Books / Children's Books / Liter...",https://www.amazon.com/dp/0008305838
4,0008375526,0008375526,Skincare: The award-winning ultimate no-nonsen...,Caroline Hirons,In Stock.,NaN,NaN,28.89,USD,4.8 out of 5 stars,10884,"[""Books"",""Crafts, Hobbies & Home"",""Home Improv...","[{""category"":""Books / Health, Fitness & Dietin...",https://www.amazon.com/dp/0008375526
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2264,B07P5BPVGM,None,Unspeakable Things,Jess Lourey,None,NaN,NaN,NaN,USD,4.2 out of 5 stars,17923,"[""Books"",""Mystery, Thriller & Suspense"",""Thril...","[{""category"":""Books / Mystery, Thriller & Susp...",https://www.amazon.com/dp/B07P5BPVGM
2265,B07P5JBCFL,None,What to Expect When You’re Expecting,"Heidi Murkoff (Author, Narrator), Meeghan Hola...",None,NaN,NaN,NaN,USD,4.8 out of 5 stars,25304,"[""Books"",""Health, Fitness & Dieting"",""Women's ...","[{""category"":""Books / Health, Fitness & Dietin...",https://www.amazon.com/dp/B07P5JBCFL
2266,B07NF7DFS2,None,The Home Edit: A Guide to Organizing and Reali...,"Clea Shearer (Author, Narrator), Joanna Teplin...",None,NaN,NaN,NaN,USD,4.7 out of 5 stars,11040,"[""Books"",""Crafts, Hobbies & Home"",""Home Improv...","[{""category"":""Audible Books & Originals / Home...",https://www.amazon.com/dp/B07NF7DFS2
2267,B07P67N918,None,The Family Upstairs: A Novel,"Lisa Jewell (Author), Tamaryn Payne (Narrator)...",None,NaN,NaN,NaN,USD,4.4 out of 5 stars,28030,"[""Books"",""Mystery, Thriller & Suspense"",""Thril...","[{""category"":""Audible Books & Originals / Myst...",https://www.amazon.com/dp/B07P67N918


In [6]:
from pyspark.sql.functions import col, trim, count, when

columns_to_check = ["ISBN10", "best_sellers_rank", "availability", "brand"]

# Conta nulos ou strings vazias para cada coluna
df_bronze.select([
    count(when(
        (col(c).isNull()) | (trim(col(c)) == ""), 
        c
    )).alias(f"vazios_em_{c}") 
    for c in columns_to_check
]).show()

+----------------+---------------------------+----------------------+---------------+
|vazios_em_ISBN10|vazios_em_best_sellers_rank|vazios_em_availability|vazios_em_brand|
+----------------+---------------------------+----------------------+---------------+
|             840|                          1|                   894|              1|
+----------------+---------------------------+----------------------+---------------+



In [7]:
from pyspark.sql.functions import lit

columns_to_fill = ["ISBN10", "best_sellers_rank", "availability", "brand"]

# Começamos com o df_bronze
df_filled_na = df_bronze

# Aplica a transformação para cada coluna na lista
for col_name in columns_to_fill:
    df_filled_na = df_filled_na.withColumn(
        col_name,
        when(
            (col(col_name).isNull()) | (trim(col(col_name)) == ""),
            lit("NA")  # Coloca a string literal "NA"
        ).otherwise(col(col_name)) # Mantém o valor original
    )

In [8]:
# Esta consulta deve agora mostrar '0' para todas as colunas
df_filled_na.select([
    count(when(
        (col(c).isNull()) | (trim(col(c)) == ""), 
        c
    )).alias(f"vazios_em_{c}") 
    for c in columns_to_check
]).show()

# Esta consulta deve mostrar as contagens do teu profiling (840, 1, 894, 1)
df_filled_na.select([
    count(when(
        col(c) == "NA", 
        c
    )).alias(f"NA_em_{c}") 
    for c in columns_to_check
]).show()

+----------------+---------------------------+----------------------+---------------+
|vazios_em_ISBN10|vazios_em_best_sellers_rank|vazios_em_availability|vazios_em_brand|
+----------------+---------------------------+----------------------+---------------+
|               0|                          0|                     0|              0|
+----------------+---------------------------+----------------------+---------------+

+------------+-----------------------+------------------+-----------+
|NA_em_ISBN10|NA_em_best_sellers_rank|NA_em_availability|NA_em_brand|
+------------+-----------------------+------------------+-----------+
|         840|                      1|               894|          1|
+------------+-----------------------+------------------+-----------+



In [9]:
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, IntegerType
from pyspark.sql.functions import from_json, col, regexp_replace

# Definir o Schema do JSON na coluna 'best_sellers_rank'. É um Array de Structs (objetos), onde cada Struct tem 'category' e 'rank'
rank_schema = ArrayType(
    StructType([
        StructField("category", StringType(), True),
        StructField("rank", StringType(), True)
    ])
)

# Aplicar a transformação
# Usamos o df_filled_na da sua célula anterior
df_rank_normalizado = df_filled_na.withColumn(
    # Ler a string JSON, ignorando as que são "NA"
    "parsed_rank",
    from_json(col("best_sellers_rank"), rank_schema)
).withColumn(
    # Extrair apenas o PRIMEIRO item do array (o rank principal)
    "primary_rank",
    col("parsed_rank")[0]
).withColumn(
    # Criar a nova coluna para a categoria do rank
    "best_seller_category",
    col("primary_rank.category")
).withColumn(
    # Criar a nova coluna para o rank
    "best_seller_rank_raw",
    col("primary_rank.rank")
)

# Limpar a coluna de rank (remover "#" e ",") e converter para número
df_rank_final = df_rank_normalizado.withColumn(
    "best_seller_rank",
    regexp_replace(col("best_seller_rank_raw"), "[#,,]", "").cast(IntegerType())
)

# Limpar o DataFrame: remover a coluna original e as colunas intermédias
df_final_normalized = df_rank_final.drop(
    "best_sellers_rank", # Remove a coluna JSON original
    "parsed_rank",       # Coluna de trabalho
    "primary_rank",      # Coluna de trabalho
    "best_seller_rank_raw" # Coluna de trabalho
)

# [IMPORTANTE] Atualize o nome do seu DataFrame de trabalho
print("Coluna 'best_sellers_rank' normalizada com sucesso.")
df_final_normalized.printSchema()
df_final_normalized.toPandas()

Coluna 'best_sellers_rank' normalizada com sucesso.
root
 |-- asin: string (nullable = true)
 |-- ISBN10: string (nullable = true)
 |-- title: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- availability: string (nullable = true)
 |-- discount: double (nullable = true)
 |-- initial_price: double (nullable = true)
 |-- final_price: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- reviews_count: integer (nullable = true)
 |-- categories: string (nullable = true)
 |-- url: string (nullable = true)
 |-- best_seller_category: string (nullable = true)
 |-- best_seller_rank: integer (nullable = true)



,asin,ISBN10,title,brand,availability,discount,initial_price,final_price,currency,rating,reviews_count,categories,url,best_seller_category,best_seller_rank
0,0007350813,0007350813,Wuthering Heights (Collins Classics),Emily Brontë,In Stock.,NaN,NaN,3.99,USD,4.6 out of 5 stars,13451,"[""Books"",""Literature & Fiction"",""Genre Fiction""]",https://www.amazon.com/dp/0007350813,Books / Literature & Fiction / Historical Fict...,28.0
1,0007513763,9780007513765,THE DAYS THE CRAYONS QUIT,Drew Daywalt,In Stock.,NaN,NaN,12.08,USD,4.8 out of 5 stars,16628,"[""Books"",""Children's Books"",""Literature & Fict...",https://www.amazon.com/dp/0007513763,Books / Children's Books / Literature & Fictio...,42.0
2,0008183988,0008183988,War Lord: Book 13 (The Last Kingdom Series),Bernard Cornwell,NA,NaN,NaN,NaN,USD,4.8 out of 5 stars,11275,"[""Books"",""Literature & Fiction"",""Genre Fiction""]",https://www.amazon.com/dp/0008183988,Books / Literature & Fiction / Historical Fict...,70.0
3,0008305838,0008305838,Code Name Bananas: The hilarious and epic new ...,David Walliams,In Stock.,NaN,NaN,20.43,USD,4.8 out of 5 stars,15520,"[""Books"",""Children's Books"",""Literature & Fict...",https://www.amazon.com/dp/0008305838,Books / Children's Books / Literature & Fictio...,99.0
4,0008375526,0008375526,Skincare: The award-winning ultimate no-nonsen...,Caroline Hirons,In Stock.,NaN,NaN,28.89,USD,4.8 out of 5 stars,10884,"[""Books"",""Crafts, Hobbies & Home"",""Home Improv...",https://www.amazon.com/dp/0008375526,"Books / Health, Fitness & Dieting / Beauty, Gr...",57.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2264,B07P5BPVGM,NA,Unspeakable Things,Jess Lourey,NA,NaN,NaN,NaN,USD,4.2 out of 5 stars,17923,"[""Books"",""Mystery, Thriller & Suspense"",""Thril...",https://www.amazon.com/dp/B07P5BPVGM,"Books / Mystery, Thriller & Suspense / Thrille...",47.0
2265,B07P5JBCFL,NA,What to Expect When You’re Expecting,"Heidi Murkoff (Author, Narrator), Meeghan Hola...",NA,NaN,NaN,NaN,USD,4.8 out of 5 stars,25304,"[""Books"",""Health, Fitness & Dieting"",""Women's ...",https://www.amazon.com/dp/B07P5JBCFL,"Books / Health, Fitness & Dieting / Women's He...",34.0
2266,B07NF7DFS2,NA,The Home Edit: A Guide to Organizing and Reali...,"Clea Shearer (Author, Narrator), Joanna Teplin...",NA,NaN,NaN,NaN,USD,4.7 out of 5 stars,11040,"[""Books"",""Crafts, Hobbies & Home"",""Home Improv...",https://www.amazon.com/dp/B07NF7DFS2,Audible Books & Originals / Home & Garden,75.0
2267,B07P67N918,NA,The Family Upstairs: A Novel,"Lisa Jewell (Author), Tamaryn Payne (Narrator)...",NA,NaN,NaN,NaN,USD,4.4 out of 5 stars,28030,"[""Books"",""Mystery, Thriller & Suspense"",""Thril...",https://www.amazon.com/dp/B07P67N918,"Audible Books & Originals / Mystery, Thriller ...",13.0


In [10]:
from pyspark.sql.functions import split, element_at, trim, col

df_com_subcategoria = df_final_normalized 

split_col = split(col("best_seller_category"), " / ")

last_element = element_at(split_col, -2)


df_com_subcategoria = df_com_subcategoria.withColumn(
    "best_seller_subcategory",
    trim(last_element)
)

df_com_subcategoria.printSchema()
df_com_subcategoria.toPandas()

root
 |-- asin: string (nullable = true)
 |-- ISBN10: string (nullable = true)
 |-- title: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- availability: string (nullable = true)
 |-- discount: double (nullable = true)
 |-- initial_price: double (nullable = true)
 |-- final_price: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- reviews_count: integer (nullable = true)
 |-- categories: string (nullable = true)
 |-- url: string (nullable = true)
 |-- best_seller_category: string (nullable = true)
 |-- best_seller_rank: integer (nullable = true)
 |-- best_seller_subcategory: string (nullable = true)



,asin,ISBN10,title,brand,availability,discount,initial_price,final_price,currency,rating,reviews_count,categories,url,best_seller_category,best_seller_rank,best_seller_subcategory
0,0007350813,0007350813,Wuthering Heights (Collins Classics),Emily Brontë,In Stock.,NaN,NaN,3.99,USD,4.6 out of 5 stars,13451,"[""Books"",""Literature & Fiction"",""Genre Fiction""]",https://www.amazon.com/dp/0007350813,Books / Literature & Fiction / Historical Fict...,28.0,Historical Fiction
1,0007513763,9780007513765,THE DAYS THE CRAYONS QUIT,Drew Daywalt,In Stock.,NaN,NaN,12.08,USD,4.8 out of 5 stars,16628,"[""Books"",""Children's Books"",""Literature & Fict...",https://www.amazon.com/dp/0007513763,Books / Children's Books / Literature & Fictio...,42.0,Christian
2,0008183988,0008183988,War Lord: Book 13 (The Last Kingdom Series),Bernard Cornwell,NA,NaN,NaN,NaN,USD,4.8 out of 5 stars,11275,"[""Books"",""Literature & Fiction"",""Genre Fiction""]",https://www.amazon.com/dp/0008183988,Books / Literature & Fiction / Historical Fict...,70.0,Historical Fiction
3,0008305838,0008305838,Code Name Bananas: The hilarious and epic new ...,David Walliams,In Stock.,NaN,NaN,20.43,USD,4.8 out of 5 stars,15520,"[""Books"",""Children's Books"",""Literature & Fict...",https://www.amazon.com/dp/0008305838,Books / Children's Books / Literature & Fictio...,99.0,Historical Fiction
4,0008375526,0008375526,Skincare: The award-winning ultimate no-nonsen...,Caroline Hirons,In Stock.,NaN,NaN,28.89,USD,4.8 out of 5 stars,10884,"[""Books"",""Crafts, Hobbies & Home"",""Home Improv...",https://www.amazon.com/dp/0008375526,"Books / Health, Fitness & Dieting / Beauty, Gr...",57.0,"Health, Fitness & Dieting"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2264,B07P5BPVGM,NA,Unspeakable Things,Jess Lourey,NA,NaN,NaN,NaN,USD,4.2 out of 5 stars,17923,"[""Books"",""Mystery, Thriller & Suspense"",""Thril...",https://www.amazon.com/dp/B07P5BPVGM,"Books / Mystery, Thriller & Suspense / Thrille...",47.0,Crime
2265,B07P5JBCFL,NA,What to Expect When You’re Expecting,"Heidi Murkoff (Author, Narrator), Meeghan Hola...",NA,NaN,NaN,NaN,USD,4.8 out of 5 stars,25304,"[""Books"",""Health, Fitness & Dieting"",""Women's ...",https://www.amazon.com/dp/B07P5JBCFL,"Books / Health, Fitness & Dieting / Women's He...",34.0,Women's Health
2266,B07NF7DFS2,NA,The Home Edit: A Guide to Organizing and Reali...,"Clea Shearer (Author, Narrator), Joanna Teplin...",NA,NaN,NaN,NaN,USD,4.7 out of 5 stars,11040,"[""Books"",""Crafts, Hobbies & Home"",""Home Improv...",https://www.amazon.com/dp/B07NF7DFS2,Audible Books & Originals / Home & Garden,75.0,Audible Books & Originals
2267,B07P67N918,NA,The Family Upstairs: A Novel,"Lisa Jewell (Author), Tamaryn Payne (Narrator)...",NA,NaN,NaN,NaN,USD,4.4 out of 5 stars,28030,"[""Books"",""Mystery, Thriller & Suspense"",""Thril...",https://www.amazon.com/dp/B07P67N918,"Audible Books & Originals / Mystery, Thriller ...",13.0,Thriller & Suspense


In [11]:
# DataFrame a ser usado (o resultado da sua célula anterior)
df_limpo = df_com_subcategoria

df_limpo = df_limpo.drop("best_seller_category")

df_limpo = df_limpo.withColumnRenamed(
    "best_seller_subcategory", 
    "best_seller_category"
)

df_limpo.printSchema()
df_limpo.toPandas()

root
 |-- asin: string (nullable = true)
 |-- ISBN10: string (nullable = true)
 |-- title: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- availability: string (nullable = true)
 |-- discount: double (nullable = true)
 |-- initial_price: double (nullable = true)
 |-- final_price: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- reviews_count: integer (nullable = true)
 |-- categories: string (nullable = true)
 |-- url: string (nullable = true)
 |-- best_seller_rank: integer (nullable = true)
 |-- best_seller_category: string (nullable = true)



,asin,ISBN10,title,brand,availability,discount,initial_price,final_price,currency,rating,reviews_count,categories,url,best_seller_rank,best_seller_category
0,0007350813,0007350813,Wuthering Heights (Collins Classics),Emily Brontë,In Stock.,NaN,NaN,3.99,USD,4.6 out of 5 stars,13451,"[""Books"",""Literature & Fiction"",""Genre Fiction""]",https://www.amazon.com/dp/0007350813,28.0,Historical Fiction
1,0007513763,9780007513765,THE DAYS THE CRAYONS QUIT,Drew Daywalt,In Stock.,NaN,NaN,12.08,USD,4.8 out of 5 stars,16628,"[""Books"",""Children's Books"",""Literature & Fict...",https://www.amazon.com/dp/0007513763,42.0,Christian
2,0008183988,0008183988,War Lord: Book 13 (The Last Kingdom Series),Bernard Cornwell,NA,NaN,NaN,NaN,USD,4.8 out of 5 stars,11275,"[""Books"",""Literature & Fiction"",""Genre Fiction""]",https://www.amazon.com/dp/0008183988,70.0,Historical Fiction
3,0008305838,0008305838,Code Name Bananas: The hilarious and epic new ...,David Walliams,In Stock.,NaN,NaN,20.43,USD,4.8 out of 5 stars,15520,"[""Books"",""Children's Books"",""Literature & Fict...",https://www.amazon.com/dp/0008305838,99.0,Historical Fiction
4,0008375526,0008375526,Skincare: The award-winning ultimate no-nonsen...,Caroline Hirons,In Stock.,NaN,NaN,28.89,USD,4.8 out of 5 stars,10884,"[""Books"",""Crafts, Hobbies & Home"",""Home Improv...",https://www.amazon.com/dp/0008375526,57.0,"Health, Fitness & Dieting"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2264,B07P5BPVGM,NA,Unspeakable Things,Jess Lourey,NA,NaN,NaN,NaN,USD,4.2 out of 5 stars,17923,"[""Books"",""Mystery, Thriller & Suspense"",""Thril...",https://www.amazon.com/dp/B07P5BPVGM,47.0,Crime
2265,B07P5JBCFL,NA,What to Expect When You’re Expecting,"Heidi Murkoff (Author, Narrator), Meeghan Hola...",NA,NaN,NaN,NaN,USD,4.8 out of 5 stars,25304,"[""Books"",""Health, Fitness & Dieting"",""Women's ...",https://www.amazon.com/dp/B07P5JBCFL,34.0,Women's Health
2266,B07NF7DFS2,NA,The Home Edit: A Guide to Organizing and Reali...,"Clea Shearer (Author, Narrator), Joanna Teplin...",NA,NaN,NaN,NaN,USD,4.7 out of 5 stars,11040,"[""Books"",""Crafts, Hobbies & Home"",""Home Improv...",https://www.amazon.com/dp/B07NF7DFS2,75.0,Audible Books & Originals
2267,B07P67N918,NA,The Family Upstairs: A Novel,"Lisa Jewell (Author), Tamaryn Payne (Narrator)...",NA,NaN,NaN,NaN,USD,4.4 out of 5 stars,28030,"[""Books"",""Mystery, Thriller & Suspense"",""Thril...",https://www.amazon.com/dp/B07P67N918,13.0,Thriller & Suspense


In [12]:
from pyspark.sql.functions import lower, trim, col, translate, regexp_replace

# Colunas de texto a limpar
text_columns_to_clean = ["title", "brand", "availability", "best_seller_category"]

# O nosso DataFrame de trabalho
df_final_strings = df_limpo

# Mapa de acentos
accent_map = {
    'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u',
    'ã': 'a', 'õ': 'o', 'â': 'a', 'ê': 'e', 'ô': 'o', 'ç': 'c',
    'ñ': 'n', 'ü': 'u', 'ë': 'e', 'ï': 'i', 'ö': 'o',
    'Á': 'A', 'É': 'E', 'Í': 'I', 'Ó': 'O', 'Ú': 'U',
    'Ã': 'A', 'Õ': 'O', 'Â': 'A', 'Ê': 'E', 'Ô': 'O', 'Ç': 'C',
    'Ñ': 'N', 'Ü': 'U', 'Ë': 'E', 'Ï': 'I', 'Ö': 'O'
}

# Converte o mapa em duas strings para a função translate()
# Esta é a forma mais eficiente de o fazer em Spark
from_chars = "".join(accent_map.keys())
to_chars = "".join(accent_map.values())


for col_name in text_columns_to_clean:

# Começa com a coluna original
    cleaned_col = col(col_name)

    # Remove espaços no início/fim (Trim)
    cleaned_col = trim(cleaned_col)

    # Remove acentos (Diacritic Folding)
    # Usamos translate() que é muito mais rápido que múltiplos regexp_replace
    cleaned_col = translate(cleaned_col, from_chars, to_chars)

    # Converte para minúsculo (Lowercase)
    cleaned_col = lower(cleaned_col)

    # Remover espaços duplos
    cleaned_col = regexp_replace(cleaned_col, " +", " ")

    # Atualiza o DataFrame com a coluna limpa
    df_final_strings = df_final_strings.withColumn(col_name, cleaned_col)

# Mostra o resultado
df_final_strings.select(text_columns_to_clean).show(truncate=False)

+------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------+-------------------------------------------+-------------------------+
|title                                                                                                             |brand                                                           |availability                               |best_seller_category     |
+------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------+-------------------------------------------+-------------------------+
|wuthering heights (collins classics)                                                                              |emily bronte                                                    |in stock.                                  |historical fiction 

In [13]:
# Save dos dados normalizados na camada Silver (Delta ou Parquet)
df_final_strings \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("hdfs://hdfs-nn:9000/projeto/silver/amazon_books")

In [14]:
df_final_strings.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("rating") \
    .option("overwriteSchema", "true") \
    .option("path", "hdfs://hdfs-nn:9000/warehouse/projeto.db/amazon_books") \
    .saveAsTable("projeto.amazon_books")

In [15]:
spark.sql(
    """
    DESCRIBE HISTORY projeto.amazon_books
    """
).show()

+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      2|2025-11-17 18:02:...|  null|    null|CREATE OR REPLACE...|{isManaged -> fal...|null|    null|     null|          1|  Serializable|        false|{numFiles -> 13, ...|        null|Apache-Spark/3.4....|
|      1|2025-11-17 16:02:...|  null|    null|CREATE OR REPLACE...|{isManaged -> fal...|null|    null|     null|          0|  Serializable|        false|{numFiles -

In [16]:
spark.sql(
    """
    SELECT * FROM projeto.amazon_books
    """
).show()

+----------+-------------+--------------------+--------------------+--------------------+--------+-------------+-----------+--------+------------------+-------------+--------------------+--------------------+----------------+--------------------+
|      asin|       ISBN10|               title|               brand|        availability|discount|initial_price|final_price|currency|            rating|reviews_count|          categories|                 url|best_seller_rank|best_seller_category|
+----------+-------------+--------------------+--------------------+--------------------+--------+-------------+-----------+--------+------------------+-------------+--------------------+--------------------+----------------+--------------------+
|0062429078|   0062429078|        pretty girls|     karin slaughter|           in stock.|    null|         null|       9.99|     USD|4.3 out of 5 stars|        10050|["Books","Mystery...|https://www.amazo...|              75|               crime|
|0143127551|

In [17]:
spark.stop()